# Step3 산불발생 선행기상 및 국지임계치 심화분석

노트북은 코드 실행과 산출물 생성만 담당한다. 해석은 `Step3_산불발생_선행기상및국지임계치_심화분석_진행예정로그.md`에 기록한다.


## S3-01. 입력·파생·누수 감사

Step2에서 확정한 산불 고유 노출과 고정 매칭 대조군을 그대로 읽고, 모든 선행 시간창이 기준시각 현재 값을 제외하는지 감사한다.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["axes.unicode_minus"] = False
try:
    plt.rcParams["font.family"] = "Malgun Gothic"
except Exception:
    pass

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "data" / "학습데이터").exists()), Path(r"D:/farm-system-public-02"))
DATA_DIR = REPO_ROOT / "data" / "학습데이터"
STEP2_TABLE_DIR = REPO_ROOT / "jsw" / "강원_재_EDA" / "outputs" / "Step2" / "tables"
STEP3_ROOT = REPO_ROOT / "jsw" / "강원_재_EDA" / "outputs" / "Step3"
TABLE_DIR = STEP3_ROOT / "tables"
PLOT_DIR = STEP3_ROOT / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

S3_ID = "S3-01"
WINDOW_HOURS = [1, 3, 6, 12, 24, 48, 72]
ROLLING_RECALC_SPECS = [
    ("직전24h_평균습도", "시점_습도_pct", 24, "mean"),
    ("직전24h_최소습도", "시점_습도_pct", 24, "min"),
    ("직전48h_평균습도", "시점_습도_pct", 48, "mean"),
    ("직전48h_최소습도", "시점_습도_pct", 48, "min"),
    ("직전24h_평균풍속", "시점_풍속_m_s", 24, "mean"),
    ("직전24h_최대풍속", "시점_풍속_m_s", 24, "max"),
    ("직전48h_평균풍속", "시점_풍속_m_s", 48, "mean"),
    ("직전48h_최대풍속", "시점_풍속_m_s", 48, "max"),
]


def read_csv_kr(path, **kwargs):
    return pd.read_csv(path, encoding="utf-8-sig", **kwargs)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_table(df, name, **kwargs):
    path = TABLE_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig", **kwargs)
    return path


def save_current_plot(name):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path

print("REPO_ROOT", REPO_ROOT)
print("STEP2_TABLE_DIR", STEP2_TABLE_DIR)
print("TABLE_DIR", TABLE_DIR)
print("PLOT_DIR", PLOT_DIR)


REPO_ROOT D:\farm-system-public-02
STEP2_TABLE_DIR D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step2\tables
TABLE_DIR D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables
PLOT_DIR D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\plots


In [2]:
# S3-01 입력: Step 2에서 확정한 산불 노출·고정 매칭 대조군만 사용한다.
paths = {
    "fire_unique": STEP2_TABLE_DIR / "S2-01_unique_weather_exposures.csv",
    "matched_controls": STEP2_TABLE_DIR / "S2-04_matched_controls.csv.gz",
    "matching_quality": STEP2_TABLE_DIR / "S2-04_matching_quality.csv",
    "matching_audit": STEP2_TABLE_DIR / "S2-04_audit_summary.csv",
    "hourly_weather": DATA_DIR / "기상_시간단위_파생.csv",
}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required S3-01 input files:\n" + "\n".join(missing))

fire_unique = read_csv_kr(paths["fire_unique"], parse_dates=["기준시각"])
matching_quality = read_csv_kr(paths["matching_quality"], parse_dates=["기준시각", "date"])
matching_audit = read_csv_kr(paths["matching_audit"])
matched_controls = read_csv_kr(paths["matched_controls"], compression="gzip", parse_dates=["기준시각", "date", "fire_기준시각", "fire_date", "control_date"])

weather_header = read_csv_kr(paths["hourly_weather"], nrows=0).columns.tolist()
required_weather_cols = ["기상셀ID", "일시", "시점_기온_C", "시점_풍속_m_s", "시점_습도_pct"]
existing_rolling_cols = [spec[0] for spec in ROLLING_RECALC_SPECS if spec[0] in weather_header]
lag_cols = [c for c in ["D-1_최소습도_pct", "D-1_평균습도_pct", "D-1_강수량합_mm", "D-2_최소습도_pct", "D-3_최소습도_pct"] if c in weather_header]
weather_usecols = [c for c in dict.fromkeys(required_weather_cols + existing_rolling_cols + lag_cols) if c in weather_header]
hourly_weather = read_csv_kr(paths["hourly_weather"], usecols=weather_usecols, parse_dates=["일시"])
hourly_weather = hourly_weather.sort_values(["기상셀ID", "일시"]).reset_index(drop=True)

print("fire_unique", fire_unique.shape)
print("matching_quality", matching_quality.shape)
print("matched_controls", matched_controls.shape)
print("hourly_weather", hourly_weather.shape)
print("rolling cols", existing_rolling_cols)
print("lag cols", lag_cols)


fire_unique (1150, 9)
matching_quality (1150, 22)
matched_controls (5632, 35)
hourly_weather (1614048, 18)
rolling cols ['직전24h_평균습도', '직전24h_최소습도', '직전48h_평균습도', '직전48h_최소습도', '직전24h_평균풍속', '직전24h_최대풍속', '직전48h_평균풍속', '직전48h_최대풍속']
lag cols ['D-1_최소습도_pct', 'D-1_평균습도_pct', 'D-1_강수량합_mm', 'D-2_최소습도_pct', 'D-3_최소습도_pct']


In [3]:
# 산불 노출과 대조군을 같은 분석 키 구조로 정규화한다.
fire_base = matching_quality.copy()
fire_base["sample_group"] = "fire"
fire_base["target"] = 1
fire_base["control_rank"] = np.nan
fire_base["analysis_id"] = fire_base["fire_exposure_id"]
fire_base["기후지형유형"] = fire_base["climate_type"]

fire_cols = [
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target",
    "기상셀ID", "기준시각", "기후지형유형", "year", "month", "hour", "date",
    "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"
]
fire_analysis = fire_base[fire_cols].copy()

control_base = matched_controls.copy()
control_base["sample_group"] = "matched_control"
control_base["target"] = 0
control_base["analysis_id"] = control_base["fire_exposure_id"].astype(str) + "_C" + control_base["control_rank"].astype(str).str.zfill(2)
if "기후지형유형" not in control_base.columns:
    control_base["기후지형유형"] = control_base.get("fire_climate_type", np.nan)
control_base = control_base.merge(
    matching_quality[["fire_exposure_id", "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"]],
    on="fire_exposure_id",
    how="left",
    suffixes=("", "_fire")
)
control_cols = [
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target",
    "기상셀ID", "기준시각", "기후지형유형", "year", "month", "hour", "date",
    "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"
]
control_analysis = control_base[control_cols].copy()

analysis_keys = pd.concat([fire_analysis, control_analysis], ignore_index=True)
analysis_keys["기준시각"] = pd.to_datetime(analysis_keys["기준시각"])
analysis_keys["date"] = pd.to_datetime(analysis_keys["date"]).dt.date.astype(str)
analysis_keys["year"] = analysis_keys["기준시각"].dt.year
analysis_keys["month"] = analysis_keys["기준시각"].dt.month
analysis_keys["hour"] = analysis_keys["기준시각"].dt.hour
analysis_keys["is_ys0071"] = analysis_keys["기상셀ID"].eq("YS_0071")
analysis_keys["is_less_than_5_matched"] = analysis_keys["matched_control_n"].fillna(0).lt(5)
analysis_keys["is_2021_feb_cluster"] = analysis_keys["기준시각"].dt.to_period("M").astype(str).eq("2021-02")
analysis_keys["is_february"] = analysis_keys["month"].eq(2)

# 기준시각에 해당하는 기존 rolling/lag 컬럼을 붙여 결측과 직접 재계산 비교에 쓴다.
point_cols = ["기상셀ID", "일시"] + existing_rolling_cols + lag_cols
point_weather = hourly_weather[point_cols].copy()
analysis_point = analysis_keys[["analysis_id", "기상셀ID", "기준시각", "sample_group", "기후지형유형"]].merge(
    point_weather,
    left_on=["기상셀ID", "기준시각"],
    right_on=["기상셀ID", "일시"],
    how="left"
)
analysis_keys["has_exact_hourly_row"] = analysis_point["일시"].notna().to_numpy()

# 시간창별 선행 원자료 관측 가능률: [기준시각-window, 기준시각)만 집계해 현재시각을 제외한다.
lead_rows = []
weather_groups = {cell: g for cell, g in hourly_weather.groupby("기상셀ID", sort=False)}
for cell, idx in analysis_keys.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups.get(cell)
    if g is None:
        for i in idx:
            for w in WINDOW_HOURS:
                lead_rows.append((analysis_keys.at[i, "analysis_id"], w, 0, False))
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    exp_times = analysis_keys.loc[idx, "기준시각"].to_numpy(dtype="datetime64[ns]")
    exp_ids = analysis_keys.loc[idx, "analysis_id"].to_numpy()
    for analysis_id, t in zip(exp_ids, exp_times):
        end = np.searchsorted(times, t, side="left")
        for w in WINDOW_HOURS:
            start = np.searchsorted(times, t - np.timedelta64(w, "h"), side="left")
            obs_n = int(end - start)
            lead_rows.append((analysis_id, w, obs_n, obs_n >= w))
lead_counts = pd.DataFrame(lead_rows, columns=["analysis_id", "window_h", "prior_obs_n", "available"])
lead_counts = lead_counts.merge(
    analysis_keys[["analysis_id", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형", "month", "hour", "fire_exposure_id"]],
    on="analysis_id",
    how="left"
)

# 기존 rolling 값이 현재시각 제외 방식과 일치하는지 직접 재계산한다.
recalc_rows = []
analysis_point_index = analysis_point.set_index("analysis_id")
for cell, idx in analysis_keys.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups.get(cell)
    if g is None:
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    for feature, source, window_h, agg in ROLLING_RECALC_SPECS:
        if feature not in hourly_weather.columns or source not in hourly_weather.columns:
            continue
        values = g[source].to_numpy(dtype="float64")
        for i in idx:
            analysis_id = analysis_keys.at[i, "analysis_id"]
            t = np.datetime64(analysis_keys.at[i, "기준시각"], "ns")
            end = np.searchsorted(times, t, side="left")
            start = np.searchsorted(times, t - np.timedelta64(window_h, "h"), side="left")
            window_values = values[start:end]
            window_values = window_values[~np.isnan(window_values)]
            recomputed = np.nan
            if len(window_values):
                if agg == "mean":
                    recomputed = float(np.mean(window_values))
                elif agg == "min":
                    recomputed = float(np.min(window_values))
                elif agg == "max":
                    recomputed = float(np.max(window_values))
            existing = analysis_point_index.at[analysis_id, feature] if analysis_id in analysis_point_index.index else np.nan
            abs_diff = np.nan if pd.isna(existing) or pd.isna(recomputed) else abs(float(existing) - recomputed)
            recalc_rows.append({
                "analysis_id": analysis_id,
                "sample_group": analysis_keys.at[i, "sample_group"],
                "기후지형유형": analysis_keys.at[i, "기후지형유형"],
                "feature": feature,
                "source_column": source,
                "window_h": window_h,
                "agg": agg,
                "prior_obs_n": int(end - start),
                "existing_value": existing,
                "recomputed_prior_only": recomputed,
                "abs_diff": abs_diff,
                "matches_prior_only": bool(pd.notna(abs_diff) and abs_diff <= 1e-9),
            })
recalc_detail = pd.DataFrame(recalc_rows)

# 표 요약 생성
input_rows = []
def add_input_row(item, value, expected=None, status=None, note=""):
    input_rows.append({"item": item, "value": value, "expected": expected, "status": status, "note": note})

add_input_row("S2 unique fire exposure rows", len(fire_unique), 1150, "ok" if len(fire_unique) == 1150 else "check")
add_input_row("S2 matching quality rows", len(matching_quality), 1150, "ok" if len(matching_quality) == 1150 else "check")
add_input_row("S2 matched control rows", len(matched_controls), 5632, "ok" if len(matched_controls) == 5632 else "check")
add_input_row("S3 analysis rows", len(analysis_keys), len(matching_quality) + len(matched_controls), "ok")
add_input_row("fire rows in S3", int((analysis_keys["target"] == 1).sum()), 1150, "ok" if int((analysis_keys["target"] == 1).sum()) == 1150 else "check")
add_input_row("control rows in S3", int((analysis_keys["target"] == 0).sum()), 5632, "ok" if int((analysis_keys["target"] == 0).sum()) == 5632 else "check")
add_input_row("full 5 matched fire exposures", int(matching_quality["full_5_matched"].sum()), 1110, "ok" if int(matching_quality["full_5_matched"].sum()) == 1110 else "check")
add_input_row("less than 5 matched fire exposures", int((~matching_quality["full_5_matched"].astype(bool)).sum()), 40, "ok" if int((~matching_quality["full_5_matched"].astype(bool)).sum()) == 40 else "check")
add_input_row("analysis key duplicate rows", int(analysis_keys.duplicated(["analysis_id"]).sum()), 0, "ok" if not analysis_keys.duplicated(["analysis_id"]).any() else "fail")
add_input_row("cell-time duplicate within sample_group", int(analysis_keys.duplicated(["sample_group", "기상셀ID", "기준시각", "fire_exposure_id", "control_rank"]).sum()), 0, "ok")
add_input_row("exact hourly row coverage pct", round(float(analysis_keys["has_exact_hourly_row"].mean() * 100), 4), 100, "ok" if analysis_keys["has_exact_hourly_row"].all() else "check")
add_input_row("S2-04 matched_controls sha256", sha256_file(paths["matched_controls"]), None, "info")
for _, row in matching_audit.iterrows():
    add_input_row(f"S2 audit: {row.get('metric') or row.iloc[0]}", row.get("value", row.iloc[-1]), None, "info")
input_audit = pd.DataFrame(input_rows)

lead_summary = (
    lead_counts.groupby(["sample_group", "기후지형유형", "window_h"], dropna=False)
    .agg(
        exposure_n=("analysis_id", "nunique"),
        available_n=("available", "sum"),
        availability_pct=("available", lambda s: float(s.mean() * 100)),
        mean_prior_obs_n=("prior_obs_n", "mean"),
        min_prior_obs_n=("prior_obs_n", "min"),
    )
    .reset_index()
)
existing_feature_rows = []
for feature in existing_rolling_cols + lag_cols:
    tmp = analysis_point[["analysis_id", "sample_group", "기후지형유형", feature]].copy()
    tmp["available"] = tmp[feature].notna()
    g = tmp.groupby(["sample_group", "기후지형유형"], dropna=False)["available"].agg(["count", "sum", "mean"]).reset_index()
    g["feature"] = feature
    g["window_h"] = np.nan
    g = g.rename(columns={"count": "exposure_n", "sum": "available_n", "mean": "availability_pct"})
    g["availability_pct"] = g["availability_pct"] * 100
    existing_feature_rows.append(g[["sample_group", "기후지형유형", "feature", "window_h", "exposure_n", "available_n", "availability_pct"]])
existing_feature_summary = pd.concat(existing_feature_rows, ignore_index=True) if existing_feature_rows else pd.DataFrame()
lead_feature_summary = lead_summary.copy()
lead_feature_summary["feature"] = "prior_hourly_obs_" + lead_feature_summary["window_h"].astype(int).astype(str) + "h"
lead_feature_summary = lead_feature_summary[["sample_group", "기후지형유형", "feature", "window_h", "exposure_n", "available_n", "availability_pct", "mean_prior_obs_n", "min_prior_obs_n"]]
lead_feature_availability = pd.concat([lead_feature_summary, existing_feature_summary], ignore_index=True, sort=False)

if recalc_detail.empty:
    leakage_recalc_audit = pd.DataFrame(columns=["feature", "sample_group", "n", "comparable_n", "match_n", "mismatch_n", "max_abs_diff", "status"])
else:
    leakage_recalc_audit = (
        recalc_detail.groupby(["feature", "sample_group"], dropna=False)
        .agg(
            n=("analysis_id", "count"),
            comparable_n=("abs_diff", lambda s: int(s.notna().sum())),
            match_n=("matches_prior_only", "sum"),
            mismatch_n=("matches_prior_only", lambda s: int((~s.astype(bool)).sum())),
            max_abs_diff=("abs_diff", "max"),
            median_abs_diff=("abs_diff", "median"),
            min_prior_obs_n=("prior_obs_n", "min"),
        )
        .reset_index()
    )
    leakage_recalc_audit["status"] = np.where(
        (leakage_recalc_audit["comparable_n"] > 0) & (leakage_recalc_audit["max_abs_diff"].fillna(0) <= 1e-9),
        "ok_prior_only_matches_existing",
        "check_mismatch_or_missing"
    )

flagged_exposures = analysis_keys[[
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형",
    "month", "hour", "date", "matched_control_n", "full_5_matched", "is_less_than_5_matched", "is_february", "is_2021_feb_cluster", "is_ys0071", "has_exact_hourly_row"
]].copy()

# 저장
written_tables = []
for df, name in [
    (input_audit, "S3-01_input_population_audit.csv"),
    (lead_feature_availability, "S3-01_lead_feature_availability.csv"),
    (leakage_recalc_audit, "S3-01_leakage_recalculation_audit.csv"),
    (recalc_detail, "S3-01_leakage_recalculation_detail.csv"),
    (flagged_exposures, "S3-01_flagged_exposures.csv"),
]:
    written_tables.append(write_table(df, name))

input_audit


,item,value,expected,status,note
0,S2 unique fire exposure rows,1150,1150.0,ok,
1,S2 matching quality rows,1150,1150.0,ok,
2,S2 matched control rows,5632,5632.0,ok,
3,S3 analysis rows,6782,6782.0,ok,
4,fire rows in S3,1150,1150.0,ok,
5,control rows in S3,5632,5632.0,ok,
6,full 5 matched fire exposures,1110,1110.0,ok,
7,less than 5 matched fire exposures,40,40.0,ok,
8,analysis key duplicate rows,0,0.0,ok,
9,cell-time duplicate within sample_group,0,0.0,ok,


In [4]:
# S3-01 플롯: 선행 원자료 관측 가능률과 기존 rolling/lag 결측을 한눈에 점검한다.
plot_df = lead_feature_availability.copy()
plot_df["feature_label"] = np.where(
    plot_df["window_h"].notna(),
    "prior_obs_" + plot_df["window_h"].fillna(0).astype(int).astype(str) + "h",
    plot_df["feature"].astype(str)
)
plot_df["group_label"] = plot_df["sample_group"].astype(str) + " | " + plot_df["기후지형유형"].astype(str)
pivot = plot_df.pivot_table(index="feature_label", columns="group_label", values="availability_pct", aggfunc="mean")

height = max(4, 0.34 * len(pivot.index) + 1.5)
width = max(8, 0.7 * len(pivot.columns) + 2)
plt.figure(figsize=(width, height))
sns.heatmap(pivot, vmin=0, vmax=100, cmap="viridis", annot=True, fmt=".1f", linewidths=0.4, linecolor="white", cbar_kws={"label": "availability (%)"})
plt.title("S3-01 lead feature availability by sample group and climate type")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=35, ha="right")
heatmap_path = save_current_plot("S3-01_feature_availability_heatmap.png")

missing_position = lead_counts.copy()
missing_position["missing"] = ~missing_position["available"]
missing_summary = (
    missing_position.groupby(["기상셀ID", "window_h"], dropna=False)
    .agg(total_n=("analysis_id", "count"), missing_n=("missing", "sum"), missing_pct=("missing", lambda s: float(s.mean() * 100)))
    .reset_index()
)
missing_pivot = missing_summary.pivot(index="기상셀ID", columns="window_h", values="missing_pct").fillna(0)
plt.figure(figsize=(7, max(5, 0.12 * len(missing_pivot.index))))
sns.heatmap(missing_pivot, vmin=0, vmax=100, cmap="mako_r", cbar_kws={"label": "missing (%)"})
plt.title("S3-01 missing prior-hour coverage by weather cell")
plt.xlabel("window (h)")
plt.ylabel("weather cell")
missing_plot_path = save_current_plot("S3-01_missing_prior_hour_coverage_by_cell.png")

written_plots = [heatmap_path, missing_plot_path]
artifact_manifest = pd.DataFrame([
    {"artifact_type": "table", "path": str(p.relative_to(REPO_ROOT)), "rows": int(pd.read_csv(p, encoding="utf-8-sig").shape[0])}
    for p in written_tables
] + [
    {"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT)), "rows": np.nan}
    for p in written_plots
])
manifest_path = write_table(artifact_manifest, "S3-01_artifact_manifest.csv")

print("written tables")
for p in written_tables + [manifest_path]:
    print(" -", p)
print("written plots")
for p in written_plots:
    print(" -", p)

print("\nInput audit status counts")
print(input_audit["status"].value_counts(dropna=False))
print("\nLead availability minimum by window")
print(lead_feature_availability[lead_feature_availability["window_h"].notna()].groupby("window_h")["availability_pct"].min())
print("\nLeakage recalculation status")
print(leakage_recalc_audit["status"].value_counts(dropna=False) if not leakage_recalc_audit.empty else "no comparable rolling columns")


written tables
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_input_population_audit.csv
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_lead_feature_availability.csv
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_leakage_recalculation_audit.csv
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_leakage_recalculation_detail.csv
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_flagged_exposures.csv
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\tables\S3-01_artifact_manifest.csv
written plots
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\plots\S3-01_feature_availability_heatmap.png
 - D:\farm-system-public-02\jsw\강원_재_EDA\outputs\Step3\plots\S3-01_missing_prior_hour_coverage_by_cell.png

Input audit status counts
status
info    19
ok      11
Name: count, dtype: int64

Lead availability minimum by window
window_h
1.0     100.000000
3.0     100.000000
6.0     100.000000
12.0 